# Kaggle avatar prototype
This notebook clones the public repository `koredeycode/kaggle-avatar-prototype` into Kaggle, starts the local FastAPI/WebSocket prototype in mock mode by default, and then exposes the browser path through Cloudflare Quick Tunnel. Public cloning needs no GitHub credential. A private clone is optional through a Kaggle Secret.

In [ ]:
import os
import secrets
import subprocess
import sys
import time
from pathlib import Path

GITHUB_REPO = os.getenv('GITHUB_REPO', 'https://github.com/koredeycode/kaggle-avatar-prototype.git').strip()
GITHUB_REF = os.getenv('GITHUB_REF', 'main').strip()
GITHUB_PRIVATE = os.getenv('GITHUB_PRIVATE', 'false').strip().lower() in {'1', 'true', 'yes'}
GITHUB_TOKEN_SECRET = os.getenv('GITHUB_TOKEN_SECRET', 'GITHUB_TOKEN').strip()
PROJECT_DIR = Path('/kaggle/working/kaggle-avatar-prototype')
if not PROJECT_DIR.exists():
    clone_env = os.environ.copy()
    if GITHUB_PRIVATE:
        from kaggle_secrets import UserSecretsClient
        github_token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET)
        askpass = Path('/tmp/avatar-github-askpass.sh')
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf \'%s\\n\' \'x-access-token\' ;;\n  *) printf \'%s\\n\' "$GITHUB_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        clone_env['GIT_ASKPASS'] = str(askpass)
        clone_env['GIT_TERMINAL_PROMPT'] = '0'
        clone_env['GITHUB_TOKEN'] = github_token
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', GITHUB_REF, GITHUB_REPO, str(PROJECT_DIR)], env=clone_env, check=True)
    clone_env.pop('GITHUB_TOKEN', None)
    if GITHUB_PRIVATE:
        askpass.unlink(missing_ok=True)
else:
    print('Using existing checkout:', PROJECT_DIR)
    update = subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', GITHUB_REF], text=True, capture_output=True)
    print(update.stdout, update.stderr)
    if update.returncode != 0:
        print('Existing checkout was not updated; continuing with its current commit.')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))
runtime_token = secrets.token_urlsafe(32)
env = os.environ.copy()
env.update({
    'APP_HOST': '127.0.0.1',
    'APP_PORT': '8000',
    'RUNTIME_TOKEN': runtime_token,
    'MODEL_MODE': 'mock',
    'TTS_MODE': 'mock',
    'VAD_MODE': 'mock',
    'PUBLIC_MODE': 'true',
})
print('Runtime token:', runtime_token)
print('Keep this token private.')

In [ ]:
import json
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
MODEL_PROFILE = os.getenv('MODEL_PROFILE', 'mock').strip().lower()
MODEL_ROOT = Path('/kaggle/working/avatar-models')
env['MODEL_ROOT'] = str(MODEL_ROOT)
env['HF_HOME'] = str(MODEL_ROOT / 'huggingface')
env.setdefault('OLLAMA_MODEL', 'qwen3:8b')
env['OLLAMA_URL'] = 'http://127.0.0.1:11434'
env['OLLAMA_HOST'] = '127.0.0.1:11434'
env['PYTHONPATH'] = str(PROJECT_DIR / 'src') + os.pathsep + env.get('PYTHONPATH', '')
ollama_process = None
if MODEL_PROFILE in {'local', 'auto'}:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[local-models]'], check=True)
        subprocess.run(['apt-get', 'update'], check=True)
        subprocess.run(['apt-get', 'install', '-y', 'zstd', 'curl', 'espeak-ng', 'libsndfile1', 'ffmpeg'], check=True)
        subprocess.run([sys.executable, 'scripts/install_ollama.py'], check=True)
        import shutil
        ollama_binary = shutil.which('ollama')
        if ollama_binary is None:
            raise RuntimeError('Ollama binary is unavailable')
        ollama_process = subprocess.Popen([ollama_binary, 'serve'], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(60):
            probe = subprocess.run([ollama_binary, 'list'], env=env, text=True, capture_output=True)
            if probe.returncode == 0:
                break
            time.sleep(1)
        if probe.returncode != 0:
            raise RuntimeError(probe.stderr or 'Ollama did not become ready')
        subprocess.run([ollama_binary, 'pull', env['OLLAMA_MODEL']], env=env, check=True)
        bootstrap = subprocess.run([sys.executable, 'scripts/bootstrap_models.py', '--root', str(MODEL_ROOT), '--sherpa', '--kokoro'], env=env, text=True, capture_output=True)
        print(bootstrap.stdout)
        print(bootstrap.stderr)
        manifest_path = MODEL_ROOT / 'model-manifest.json'
        model_manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
        sherpa_dir = Path(model_manifest['sherpa_dir']) if model_manifest.get('sherpa_dir') else None
        sherpa_files = [
            sherpa_dir / 'encoder-epoch-99-avg-1-chunk-16-left-128.int8.onnx',
            sherpa_dir / 'decoder-epoch-99-avg-1-chunk-16-left-128.onnx',
            sherpa_dir / 'joiner-epoch-99-avg-1-chunk-16-left-128.int8.onnx',
            sherpa_dir / 'tokens.txt',
        ] if sherpa_dir else []
        model_ok = bootstrap.returncode == 0 and sherpa_dir is not None and all(path.exists() for path in sherpa_files)
        if model_ok:
            env['SHERPA_ENCODER'] = str(sherpa_files[0])
            env['SHERPA_DECODER'] = str(sherpa_files[1])
            env['SHERPA_JOINER'] = str(sherpa_files[2])
            env['SHERPA_TOKENS'] = str(sherpa_files[3])
            env['MODEL_MODE'] = 'local'
            env['TTS_MODE'] = 'kokoro'
        else:
            raise RuntimeError('required local model assets are unavailable')
    except Exception as exc:
        print('Local model bootstrap failed; using mock profile:', exc)
        if ollama_process is not None and ollama_process.poll() is None:
            ollama_process.terminate()
        env.update({'MODEL_MODE': 'mock', 'TTS_MODE': 'mock'})
else:
    print('Using reliable mock profile; set MODEL_PROFILE=local to opt into model downloads.')
env.setdefault('VAD_MODE', 'mock')
env['VAD_MODE'] = 'mock'
env['SMART_TURN_MODE'] = 'manual'
print('Active model profile:', env['MODEL_MODE'])

In [ ]:
preflight = subprocess.run([sys.executable, 'scripts/preflight.py'], env=env, text=True, capture_output=True, check=False)
print(preflight.stdout)
print(preflight.stderr)
assert preflight.returncode == 0

In [ ]:
if 'app_process' in globals() and app_process.poll() is None:
    app_process.terminate()
    try:
        app_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        app_process.kill()
if 'app_log' in globals() and not app_log.closed:
    app_log.close()
app_log = open('/tmp/avatar-prototype-app.log', 'w', encoding='utf-8')
app_process = subprocess.Popen([sys.executable, '-m', 'avatar_prototype.main'], env=env, stdout=app_log, stderr=subprocess.STDOUT, cwd=PROJECT_DIR)
time.sleep(3)
if app_process.poll() is not None:
    print('avatar app exited early')
    print(Path('/tmp/avatar-prototype-app.log').read_text(encoding='utf-8', errors='replace'))
    raise RuntimeError('avatar app exited early; see the application log above')
print('App PID:', app_process.pid)
print('Local URL: http://127.0.0.1:8000')

In [ ]:
import httpx
health = httpx.get('http://127.0.0.1:8000/healthz', timeout=10, trust_env=False)
print(health.status_code, health.json())
health.raise_for_status()
smoke = subprocess.run([sys.executable, 'scripts/smoke_test.py', '--url', 'http://127.0.0.1:8000', '--token', runtime_token], env=env, text=True, capture_output=True)
print(smoke.stdout)
print(smoke.stderr)
assert smoke.returncode == 0, 'text response smoke test failed'

## Cloudflare browser ingress
Cloudflare Quick Tunnel is required for the browser path. It is public, temporary, and not production hosting. The tunnel exposes only FastAPI and must be stopped before the session ends.

In [ ]:
import re
import httpx
cloudflared_setup = subprocess.run([sys.executable, 'scripts/install_cloudflared.py'], env=env, text=True, capture_output=True, check=False)
print(cloudflared_setup.stdout)
print(cloudflared_setup.stderr)
assert cloudflared_setup.returncode == 0
if 'tunnel_process' in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate()
    try:
        tunnel_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        tunnel_process.kill()
if 'tunnel_log' in globals() and not tunnel_log.closed:
    tunnel_log.close()
tunnel_log = open('/tmp/avatar-prototype-tunnel.log', 'w', encoding='utf-8')
tunnel_process = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'], stdout=tunnel_log, stderr=subprocess.STDOUT, cwd=PROJECT_DIR)
tunnel_url = None
for _ in range(60):
    time.sleep(1)
    tunnel_log.flush()
    tunnel_text = Path('/tmp/avatar-prototype-tunnel.log').read_text(encoding='utf-8', errors='replace')
    match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', tunnel_text)
    if match:
        tunnel_url = match.group(0)
        break
    if tunnel_process.poll() is not None:
        raise RuntimeError(tunnel_text or 'cloudflared exited early')
assert tunnel_url is not None, 'Cloudflare did not issue a public URL'
public_health = httpx.get(f'{tunnel_url}/healthz', timeout=20)
print(public_health.status_code, public_health.json())
public_health.raise_for_status()
print('Tunnel PID:', tunnel_process.pid)
print('Public URL:', tunnel_url)

In [ ]:
CLEANUP_ON_RUN_ALL = False
def cleanup():
    for process in (tunnel_process, app_process, ollama_process):
        if process is not None and process.poll() is None:
            process.terminate()
    for process in (tunnel_process, app_process, ollama_process):
        if process is None:
            continue
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
    app_log.close()
    tunnel_log.close()
if CLEANUP_ON_RUN_ALL:
    cleanup()